# Stochastic processes for equity and rates MC

This notebook surveys the **process parameter objects** exposed in `finstack.monte_carlo`: what each parameter means, when the model is appropriate, and how to inspect them from Python.

## Concept

A **stochastic process** describes how state variables evolve in continuous time (often under a risk-neutral measure for pricing). In Monte Carlo you **discretize time**, draw random shocks, and simulate paths. **GBM** is the Black–Scholes workhorse; **Heston** adds stochastic volatility; **CIR** models positive short rates or variance; **jump** models add discontinuous moves; **Schwartz–Smith** is a two-factor commodity spot model; **multi-GBM** couples correlated underlyings.

## API walkthrough

`GbmProcess`, `HestonProcess`, and related types are lightweight **parameter bundles** (constructors + getters). High-level **GBM** pricing in Python typically goes through `EuropeanPricer` or `McEngine`, which assume GBM unless you extend lower-level Rust APIs.

Below: build **GBM** and **Heston**, print the **Feller** check for Heston variance, and run a tiny **`McEngine`** European call under GBM.

In [ ]:
from finstack.monte_carlo import (
    GbmProcess,
    HestonProcess,
    McEngine,
    TimeGrid,
    EuropeanPricer,
    black_scholes_call,
)

gbm = GbmProcess(rate=0.05, div_yield=0.0, vol=0.20)
heston = HestonProcess(
    rate=0.05,
    div_yield=0.0,
    v0=0.04,
    kappa=2.0,
    theta=0.04,
    xi=0.3,
    rho=-0.7,
)
print(f"GBM: {gbm}")
print(f"Heston: {heston}")
print(f"Heston Feller: {heston.satisfies_feller}")

grid = TimeGrid(t_max=1.0, num_steps=252)
engine = McEngine(num_paths=25_000, time_grid=grid, seed=42)
mc_call = engine.price_european_call(100.0, 100.0, 0.05, 0.0, 0.20)
bs = black_scholes_call(100.0, 100.0, 0.05, 0.0, 0.20, 1.0)
print(f"McEngine ATM call mean: {mc_call.mean.amount:.6f} (stderr={mc_call.stderr:.6f})")
print(f"Black–Scholes call:       {bs:.6f}")

pr = EuropeanPricer(num_paths=25_000, seed=42)
eu = pr.price_call(100.0, 100.0, 0.05, 0.0, 0.20, 1.0, num_steps=252)
print(f"EuropeanPricer ATM call:  {eu.mean.amount:.6f}")


## Examples

Try additional processes behind `try`/`except` so the notebook stays runnable if a symbol or signature changes. Each block prints a one-line **interpretation** and the **`repr`**.

In [ ]:
def _try(name, fn):
    try:
        obj = fn()
        print(f"[ok] {name}: {obj}")
        return True
    except Exception as exc:
        print(f"[skip] {name}: {type(exc).__name__}: {exc}")
        return False

_try("BrownianProcess", lambda: __import__("finstack.monte_carlo", fromlist=["BrownianProcess"]).BrownianProcess(0.01, 0.25))
try:
    from finstack.monte_carlo import CirProcess

    _cir = CirProcess(0.5, 0.04, 0.08, 0.03)
    print(f"[ok] CirProcess: {_cir}")
    print(f"CIR Feller satisfied: {_cir.satisfies_feller}")
except Exception as exc:
    print(f"[skip] CirProcess: {type(exc).__name__}: {exc}")

_try(
    "MertonJumpProcess",
    lambda: __import__("finstack.monte_carlo", fromlist=["MertonJumpProcess"]).MertonJumpProcess(
        0.05, 0.0, 0.18, 0.6, -0.02, 0.12
    ),
)
_try(
    "BatesProcess",
    lambda: __import__("finstack.monte_carlo", fromlist=["BatesProcess"]).BatesProcess(
        0.05, 0.0, 0.04, 2.0, 0.04, 0.3, -0.7, 0.4, -0.02, 0.1
    ),
)
_try(
    "SchwartzSmithProcess",
    lambda: __import__("finstack.monte_carlo", fromlist=["SchwartzSmithProcess"]).SchwartzSmithProcess(
        1.0, 0.2, 0.1, 0.3, 0.05, 0.02
    ),
)
rho = 0.35
corr = [1.0, rho, rho, 1.0]
_try(
    "MultiGbmProcess",
    lambda: __import__("finstack.monte_carlo", fromlist=["MultiGbmProcess"]).MultiGbmProcess(
        [0.03, 0.03], [0.0, 0.01], [0.22, 0.28], corr
    ),
)

print("— Parameter cheat-sheet —")
print("GBM: log-normal spot; use vanilla equity FX when vol is constant.")
print("Heston: stochastic variance + skew via rho; equity smiles.")
print("Brownian (arithmetic): local vol / rate building blocks; can go negative.")
print("CIR: positive mean-reverting process; short rates or variance in other models.")
print("Merton/Bates: jumps (events) and Bates = Heston + jumps for short-dated skew.")
print("Schwartz–Smith: two-factor log-spot decomposition; commodity forward curves.")
print("Multi-GBM: baskets/spread options with correlated diffusions.")


## Takeaways

- **GBM** is the default workhorse for equity-style Monte Carlo in this stack; **`McEngine`** prices Europeans under GBM with an **exact** time stepper.
- **Heston** parameters are all explicit in Python; check **`satisfies_feller`** to know if variance can hit zero in continuous time.
- **CIR** has its own **Feller** condition for positivity.
- **Jumps** (Merton, Bates) matter when short-tenor returns show **fat tails**; **Bates** combines stochastic vol and jumps.
- **Schwartz–Smith** and **multi-GBM** are the natural starting points for **commodity** and **correlation** structures, respectively.